In [ ]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

In [ ]:
import torch.nn as nn
import torch
from seqAE_model import SeqAutoencoder
from contra_seq_dataset import *
from torch.utils.data import DataLoader, RandomSampler
from tqdm import tqdm

def get_latents(model_tag, csv_name, load_bs=32):
    
    tag = model_tag
    bs = load_bs

    n_epochs = 30
    use_cuda = True
    empty_cuda = True
    cuda_ids = [0,1,2,3]
    model = SeqAutoencoder(dim_emb=512, heads=8, dim_hidden=32,
                           L_enc=6, L_dec=6, dim_ff=2048, 
                           drpt=0.1, actv='relu', eps=0.6, b_first=True)

    p = f'/home/kat/Repos/SALSA/results/models/{tag}/{n_epochs-1:02}.pt'
    model.load_state_dict(torch.load(p), strict = False)
    if empty_cuda:
        torch.cuda.empty_cache()
    if use_cuda:
        if len(cuda_ids) == 1:
            cuda_id = cuda_ids[0]
            device = torch.device(f"cuda:{cuda_id}")
        elif len(cuda_ids) > 1:
            device =  torch.device("cuda")
            print(f"Using {len(cuda_ids)} GPUs!")
            model = nn.DataParallel(model, device_ids=cuda_ids)
            model.to(device)
    else:
        device = torch.device("cpu")
        model = model.to(device)
    model = model.eval()
    print(f"Loaded model weights from {p}!")

    ds = ContraSeqDataset(csv_name)
    df = get_dataset_array(csv_name)

    loader = DataLoader(ds, batch_size=bs, sampler=range(len(df)), 
                        num_workers=0, pin_memory=True)
    latents = []
    for samp in tqdm(loader, total=len(df)//bs):
        for k,v in samp.items():
            if torch.is_tensor(v):
                samp[k] = v.to(device)
        latent, _ = model.forward(samp['seq'], samp['pad_mask'], 
                                  samp['avg_mask'], samp['out_mask'])
        latent = latent.cpu().detach().numpy()
        latents.append(latent)
    latents = np.concatenate(latents, axis=0)
    
    return latents

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances
from scipy.spatial import distance

df = pd.read_csv('20220603_extended_augset_new.csv')

tags = ['2022041804_04',  # salsa
        '2022041807_a03', # contrastive  
        '2022041809_a04'] # vanilla ae
tag_to_dists = {}

for tag in tags:
    latents = get_latents(tag, csv_name='20220603_extended_augset_new.csv', load_bs=32)
    
    depths = [1,2,3]
    depth_to_dists = {}
    
    for depth in depths:

        depth_idc = df.index[[df.aug_iter==depth]].tolist()
        df_depth = df.loc[depth_idc]
        print(df_depth.shape, latents.shape)

        dists = []
        for i in df_depth.anc_idx.unique():

            anc_latent = latents[i]
            aug_idc = df_depth.index[[df_depth.anc_idx==i]].tolist() 
            aug_latents = latents[aug_idc]

            for aug_latent in aug_latents:
                d = distance.euclidean(anc_latent, aug_latent)
                dists.append(d)
                
        depth_to_dists[depth] = dists
        
    if tag=='2022041804_04':
        tag_to_dists['salsa'] = depth_to_dists
    elif tag=='2022041807_a03':
        tag_to_dists['contrastive encoder'] = depth_to_dists
    elif tag=='2022041809_a04':
        tag_to_dists['vanilla AE'] = depth_to_dists        

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='ticks',font_scale=1.5,palette='muted',)



for k,v in tag_to_dists.items():
    plt.figure(figsize=(20,5))
    sns.histplot(v, fill=False, element="step", kde=True, palette='magma')
#     sns.distplot(v, hist=False, kde=True)
    
    # sns.displot(data=depth_to_dists, kind="kde")

    plt.xlabel('Anc-aug Euclidean distance')
    plt.title(k)
    plt.show() 

In [ ]:
things = {}

for k,v in tag_to_dists.items():
    for k1,v1 in v.items():
        things[k] = v1

In [ ]:
things['contrastive encoder']

In [ ]:
things['salsa']

In [ ]:

plt.figure(figsize=(20,5))
sns.histplot(things, fill=False, element="step", kde=False, palette='magma')
plt.xlabel('Anc-aug Euclidean distance')
plt.show()

In [ ]:
all_dists = [x for x in tag_to_dists.values()]
# all_dists = {k:v for k,v in all_dists[0].items()}

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='ticks',font_scale=1.5,palette='muted')

for k in all_dists.keys():
    

    plt.figure(figsize=(20,5))
    sns.histplot(all_dists[k], fill=False, element="step", kde=False, palette='magma')
    plt.xlabel('Anc-aug Euclidean distance')
    plt.show()

In [ ]:


import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='ticks',font_scale=1.5,palette='muted')

plt.figure(figsize=(20,5))
sns.histplot(all_dists, fill=False, element="step", kde=False, palette='magma')
plt.xlabel('Anc-aug Euclidean distance')
plt.show()

In [ ]:
# # # # # # # # #
# depth = 1
# # # # # # # # #

# depth_idc = df.index[[df.aug_iter==depth]].tolist()

# df_depth = df.loc[depth_idc]
# latents = latents[depth_idc]
# print(len(df_depth), latents.shape)

In [ ]:
# from sklearn.metrics.pairwise import euclidean_distances
# from scipy.spatial import distance

# dists = []
# for i in df_depth.anc_idx.unique():
#     anc_latent = latents[i]
#     aug_idc = df_depth.index[[df_depth.anc_idx==i]].tolist()    
#     aug_latents = latents[aug_idc]
    
#     for aug_latent in aug_latents:
#         d = distance.euclidean(anc_latent, aug_latent)
#         dists.append(d)